In [1]:
import torch
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from transformers import CamembertTokenizer, CamembertModel
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


c:\Users\abdou\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:

# Télécharger les stopwords français
nltk.download("stopwords")
stop_words = set(stopwords.words("french"))

# Charger les données
file_path = "Data/intent-detection-extended-v2.csv"  # Remplace par le bon chemin
df = pd.read_csv(file_path)

# Nettoyage du texte
def preprocess_text(text):
    text = text.lower()  # Minuscule
    text = text.translate(str.maketrans("", "", string.punctuation))  # Supprime ponctuation
    text = " ".join([word for word in text.split() if word not in stop_words])  # Supprime stopwords
    return text

df["cleaned_text"] = df["text"].apply(preprocess_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdou\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:

# Initialiser le tokenizer et le modèle Camembert
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
model = CamembertModel.from_pretrained("camembert-base")


In [5]:

# Fonction pour obtenir les embeddings
def get_embeddings(text):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.no_grad():  # Désactiver le calcul des gradients pour accélérer
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # Prendre l'embedding du token [CLS]

# Générer les embeddings pour tous les textes
df["embeddings"] = df["cleaned_text"].apply(get_embeddings)


In [6]:

# Convertir en format utilisable par SVM
X = np.vstack(df["embeddings"].values)
y = df["label"]

# Encoder les labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in splitter.split(X, y_encoded):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

# # Séparation train/test
# X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


In [7]:

# Entraînement du modèle SVM
svm_model = SVC(kernel="linear", probability=True)  # Activer probability=True pour récupérer les scores de confiance
svm_model.fit(X_train, y_train)


SVC(kernel='linear', probability=True)

In [8]:

# Prédictions
y_pred = svm_model.predict(X_test)


In [9]:

# Évaluation du modèle
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


Accuracy: 0.75
                   precision    recall  f1-score   support

      book_flight       0.40      1.00      0.57         2
       book_hotel       0.40      0.67      0.50         3
         carry_on       1.00      1.00      1.00         3
    flight_status       1.00      0.33      0.50         3
     lost_luggage       1.00      0.67      0.80         3
     out_of_scope       1.00      0.50      0.67         6
        translate       1.00      1.00      1.00         3
     travel_alert       1.00      1.00      1.00         2
travel_suggestion       0.75      1.00      0.86         3

         accuracy                           0.75        28
        macro avg       0.84      0.80      0.77        28
     weighted avg       0.87      0.75      0.75        28



In [10]:

# Fonction de prédiction avec gestion de la classe inconnue
def predict_with_threshold(text, threshold=0.5):
    embedding = get_embeddings(preprocess_text(text)).reshape(1, -1)
    probs = svm_model.predict_proba(embedding)
    max_prob = np.max(probs)
    predicted_label = np.argmax(probs)

    if max_prob < threshold:
        return "out_of_scope"  # Classe inconnue
    return label_encoder.inverse_transform([predicted_label])[0]




In [11]:
# Exemple d'utilisation
texte_test = "traduit cela en anglais ?"
print("Prédiction :", predict_with_threshold(texte_test, threshold=0.5))

Prédiction : translate


In [37]:
# Exemple d'utilisation
texte_test = "où je dois mettre mon baggage ?"
print("Prédiction :", predict_with_threshold(texte_test, threshold=0.2))

Prédiction : carry_on


In [40]:
texte_test = "j'ai perdu mon baggage?"
print("Prédiction :", predict_with_threshold(texte_test, threshold=0.5))

Prédiction : lost_luggage


In [43]:
texte_test = "quand est le vol vers casablanca ?"
print("Prédiction :", predict_with_threshold(texte_test, threshold=0.3))

Prédiction : flight_status


In [45]:
texte_test = "comment reserver un hotel à Bali ?"
print("Prédiction :", predict_with_threshold(texte_test, threshold=0.1))

Prédiction : book_flight


In [25]:
def evaluate_model(predict_with_threshold, csv_path, threshold=0.5):
    # Charger les données minimales
    df = pd.read_csv(csv_path)
    
    # Appliquer la fonction de prédiction sur chaque texte
    df['predicted_label'] = df['text'].apply(lambda x: predict_with_threshold(x, threshold))
    
    # Calculer la précision
    accuracy = accuracy_score(df['label'], df['predicted_label'])
    
    # Afficher les résultats
    print("Précision du modèle :", accuracy)
    
    return df

# Exemple d'utilisation
csv_path = "intent-detection-minimal.csv"

In [67]:
evaluate_model(predict_with_threshold, csv_path, threshold=0.2)

Précision du modèle : 1.0


,text,label,predicted_label
0,Comment dit-on 'bonjour' en espagnol ?,translate,translate
1,Peux-tu traduire cette phrase en allemand ?,translate,translate
2,Y a-t-il des restrictions de voyage pour le Br...,travel_alert,travel_alert
3,Mon pays de destination est-il sous alerte rou...,travel_alert,travel_alert
4,Peux-tu vérifier si mon vol AF456 est à l'heure ?,flight_status,flight_status
5,Est-ce que le vol pour Madrid est retardé ?,flight_status,flight_status
6,"Ma valise a disparu à l'aéroport, comment la r...",lost_luggage,lost_luggage
7,"J'ai perdu mes bagages, que dois-je faire ?",lost_luggage,lost_luggage
8,As-tu une idée de voyage pour un week-end en E...,travel_suggestion,travel_suggestion
9,Quelle est la meilleure destination pour un vo...,travel_suggestion,travel_suggestion


In [68]:
svm_model

SVC(kernel='linear', probability=True)

In [69]:
label_encoder

LabelEncoder()

In [70]:
import joblib

In [71]:
joblib.dump(svm_model, "intent_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Modèle et encodeur sauvegardés avec succès.")

Modèle et encodeur sauvegardés avec succès.


In [17]:
import pandas as pd
import numpy as np
import torch
from transformers import CamembertTokenizer, CamembertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Charger les données
file_path = "Data/intent-detection-extended-v2.csv"
df = pd.read_csv(file_path)

# Séparer les classes connues et inconnues
df_known = df[df["label"] != "out_scope"]
df_unknown = df[df["label"] == "out_scope"]

# Encoder les labels connus
label_encoder = LabelEncoder()
df_known["label_encoded"] = label_encoder.fit_transform(df_known["label"])
num_classes = len(label_encoder.classes_)


X = df_known["text"]
y_encoded = df_known["label_encoded"]

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, test_idx in splitter.split(X, y_encoded):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
# # Séparation train/test
# X_train, X_test, y_train, y_test = train_test_split(
#     df_known["text"], df_known["label_encoded"], test_size=0.2, random_state=42
# )

# Initialiser le tokenizer
tokenizer = CamembertTokenizer.from_pretrained("camembert-base")

# Créer une classe Dataset
class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Créer les datasets et dataloaders
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = IntentDataset(X_train, y_train, tokenizer)
test_dataset = IntentDataset(X_test, y_test, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Charger le modèle
model = CamembertForSequenceClassification.from_pretrained("camembert-base", num_labels=num_classes).to(dev)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Entraînement
def train(model, train_loader, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(dev)
            attention_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}")

train(model, train_loader, optimizer)

# Évaluation
def evaluate(model, test_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(dev)
            attention_mask = batch["attention_mask"].to(dev)
            labels = batch["labels"].to(dev)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(F.softmax(logits, dim=1), axis=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    print("Accuracy:", accuracy_score(true_labels, predictions))
    print(classification_report(true_labels, predictions, target_names=label_encoder.classes_))

evaluate(model, test_loader)

# Fonction de prédiction
def predict_with_threshold(text, threshold=0.5):
    model.eval()
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    input_ids = encoding["input_ids"].to(dev)
    attention_mask = encoding["attention_mask"].to(dev)
    with torch.no_grad():
        logits = model(input_ids, attention_mask=attention_mask).logits
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        max_prob = np.max(probs)
        predicted_label = np.argmax(probs)
        if max_prob < threshold:
            return "out_scope"
        return label_encoder.inverse_transform([predicted_label])[0]

# Exemple d'utilisation
texte_test = "Quel est le statut de ma commande ?"
print("Prédiction :", predict_with_threshold(texte_test))


Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1, Loss: 2.1834261076790944
Epoch 2, Loss: 2.1384100232805525
Epoch 3, Loss: 2.0722210577556064
Epoch 4, Loss: 1.9878116931234087
Epoch 5, Loss: 1.875529135976519
Epoch 6, Loss: 1.7466968723705836
Epoch 7, Loss: 1.6020181775093079
Epoch 8, Loss: 1.467425789151873
Epoch 9, Loss: 1.342900310243879
Epoch 10, Loss: 1.2269158107893807
Accuracy: 0.8928571428571429
                   precision    recall  f1-score   support

      book_flight       1.00      1.00      1.00         2
       book_hotel       0.75      1.00      0.86         3
         carry_on       1.00      1.00      1.00         3
    flight_status       1.00      1.00      1.00         3
     lost_luggage       1.00      1.00      1.00         3
     out_of_scope       0.80      0.67      0.73         6
        translate       1.00      1.00      1.00         3
     travel_alert       1.00      1.00      1.00         2
travel_suggestion       0.67      0.67      0.67         3

         accuracy                        

In [19]:
model_path = "Models/model_30.pth"
torch.save(model.state_dict(), model_path)
print("Modèle sauvegardé.")

Modèle sauvegardé.


In [21]:
# Exemple d'utilisation
texte_test = "comment reserver un airbnb à Bali ?"
print("Prédiction :", predict_with_threshold(texte_test, threshold=0.2))

Prédiction : book_hotel


In [23]:
label_encoder.classes_

array(['book_flight', 'book_hotel', 'carry_on', 'flight_status',
       'lost_luggage', 'out_of_scope', 'translate', 'travel_alert',
       'travel_suggestion'], dtype=object)

In [30]:
csv_path = "Data/intent-detection-minimal.csv"
evaluate_model(predict_with_threshold, csv_path, threshold=0.2)

Précision du modèle : 1.0


,text,label,predicted_label
0,Comment dit-on 'bonjour' en espagnol ?,translate,translate
1,Peux-tu traduire cette phrase en allemand ?,translate,translate
2,Y a-t-il des restrictions de voyage pour le Br...,travel_alert,travel_alert
3,Mon pays de destination est-il sous alerte rou...,travel_alert,travel_alert
4,Peux-tu vérifier si mon vol AF456 est à l'heure ?,flight_status,flight_status
5,Est-ce que le vol pour Madrid est retardé ?,flight_status,flight_status
6,"Ma valise a disparu à l'aéroport, comment la r...",lost_luggage,lost_luggage
7,"J'ai perdu mes bagages, que dois-je faire ?",lost_luggage,lost_luggage
8,As-tu une idée de voyage pour un week-end en E...,travel_suggestion,travel_suggestion
9,Quelle est la meilleure destination pour un vo...,travel_suggestion,travel_suggestion
